## Import packages

In [1]:
import muse_sc as muse
import simulation_tool.multi_modal_simulation as simulation

import phenograph
from sklearn.decomposition import PCA
import numpy as np
from sklearn.metrics.cluster import adjusted_rand_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import pandas as pd
from os.path import join

# import tensorflow as tf
# tf.get_logger().setLevel('ERROR')
# np.random.seed(0)

## Generate simulation data


Simulation parameters

In [2]:
latent_dim = 100
num_cluster = 9
sample_size = 1000
latent_code_dim = 30
observed_data_dim = 500
sigma_1 = 0.1  
sigma_2 = 0.1
decay_coef_1 = 0.5 
decay_coef_2 = 0.1
merge_prob = 0.7



Use simulation tool to generate multi-modality data

In [3]:

# dataset = 'section2'
# data_dir = f'/home/lqr/CLIMA/datasets/{dataset}'
# dataset = 'Mouse_brain_hippocampus_STexpr_cellSegmentation'
dataset = 'test_1000'
data_dir = f'/home/lqr/CLIMA/datasets/{dataset}'

data_a = pd.read_csv(join(data_dir, 'cell_st_rna_data.csv'), index_col=0)
data_b = pd.read_csv(join(data_dir, 'new_cell_deep_feature.csv'), index_col=0)
data_a = data_a.values
data_b = data_b.values

In [4]:
# data = simulation.multi_modal_simulator(num_cluster, sample_size,
#                                         observed_data_dim, observed_data_dim,
#                                         latent_code_dim,
#                                         sigma_1, sigma_2,
#                                         decay_coef_1, decay_coef_2,
#                                         merge_prob)
# data_a = data['data_a_dropout']
# data_b = data['data_b_dropout']
# label_a = data['data_a_label']
# label_b = data['data_b_label']
# label_true = data['true_cluster']

In [5]:
data_a.shape

(43757, 16572)

## Analyses based on single modality

Learn features from single modality

In [6]:
view_a_feature = PCA(n_components=latent_dim, random_state=42).fit_transform(data_a)
view_b_feature = PCA(n_components=latent_dim, random_state=42).fit_transform(data_b)

Perform clustering using PhenoGraph

In [7]:
# view_a_label, _, _ = phenograph.cluster(view_a_feature)
# view_b_label, _, _ = phenograph.cluster(view_b_feature)
view_a_label = KMeans(n_clusters=num_cluster, random_state=42).fit_predict(view_a_feature)
view_b_label = KMeans(n_clusters=num_cluster, random_state=42).fit_predict(view_b_feature)

## Combined analysis using MUSE

MUSE learns the joint latent representation

In [8]:
muse_feature, reconstruct_x, reconstruct_y, \
latent_x, latent_y = muse.muse_fit_predict(data_a,
                                           data_b,
                                           view_a_label,
                                           view_b_label,
                                           latent_dim=100,
                                           n_epochs=500,
                                           weight_penalty=5,
                                           triplet_lambda=5,
                                           n_cluster=num_cluster)

++++++++++ MUSE for multi-modality single-cell analysis ++++++++++
epoch: 0, 	 total loss: nan,	 reconstruction loss: nan,	 sparse penalty: nan


KeyboardInterrupt: 

## Perform clustering
PhenoGraph clustering

In [ ]:
# muse_label, _, _ = phenograph.cluster(muse_feature)
muse_label = KMeans(n_clusters=num_cluster, random_state=42).fit_predict(muse_feature)

## Visualization of latent spaces 
Latent spaces of single-modality features or MUSE features were visualized using tSNE, with ground truth cluster labels.

Cluster accuries were quantified using adjusted Rand index (ARI). ARI = 1 indicates perfectly discover true cell identities.

In [ ]:
label_true = view_a_label

In [ ]:
plt.figure(figsize=(17, 5))
plt.subplot(1, 3, 1)
X_embedded = TSNE(n_components=2).fit_transform(view_a_feature)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    plt.title('Transcript-alone, ARI = %01.3f' % adjusted_rand_score(label_true, view_a_label))

plt.subplot(1, 3, 2)
X_embedded = TSNE(n_components=2).fit_transform(view_b_feature)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    plt.title('Morphology-alone, ARI = %01.3f' % adjusted_rand_score(label_true, view_b_label))

plt.subplot(1, 3, 3)
X_embedded = TSNE(n_components=2).fit_transform(muse_feature)
for i in np.unique(label_true):
    idx = np.nonzero(label_true == i)[0]
    plt.scatter(X_embedded[idx, 0], X_embedded[idx, 1])
    plt.title('MUSE, ARI = %01.3f' % adjusted_rand_score(label_true, muse_label))

In [ ]:
label = muse_label

label_df = pd.DataFrame(label, columns=['type'])
cell_id = pd.read_csv(join(data_dir, 'cell_id.csv'))
cell_id_name = cell_id["cell_id"].tolist()
label_df.index = cell_id_name[:10000]
label_df.reset_index(inplace=True, names='cell_id')

In [ ]:
label_df.head()

In [ ]:
label_df.to_csv(join(data_dir, f'/home/lqr/CLIMA/final_res/{dataset}/sum_spot_cell_type.csv'), index=False)